<a href="https://colab.research.google.com/github/Nafiz-kodar/Machine_learning_project/blob/main/Copy_of_cleaned_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
#path = '/Users/nafizahmed/Library/CloudStorage/GoogleDrive-nafizahmednafi@gmail.com/Other computers/My Computer/BRACU Books and Classnotes/3rd year/7th sem/cse422/Updated_Obesity_Dataset.csv'
path='/content/Updated_Obesity_Dataset.csv'
df = pd.read_csv(path)
df.head()
df_clean = df.copy()

# 1. Check for duplicates
print("Total duplicates:", df_clean.duplicated().sum())
df_clean = df_clean.drop_duplicates()

# 2. Handle missing values - let's examine them first
print("\nMissing values after removing duplicates:")
print(df_clean.isnull().sum())

# Check specific rows with missing values
print("\nRows with missing values:")
for col in df_clean.columns:
    if df_clean[col].isnull().sum() > 0:
        print(f"\nColumn: {col}")
        missing_indices = df_clean[df_clean[col].isnull()].index.tolist()
        print(f"Missing count: {len(missing_indices)}")
        if len(missing_indices) < 10:  # Show sample if not too many
            print(f"Sample indices: {missing_indices[:10]}")

# 3. Clean specific columns

# For Gender column - replace empty strings with NaN then maybe mode
df_clean['Gender'] = df_clean['Gender'].replace('', pd.NA)

# For CH2O (water intake) - looks like numerical, but has some missing values
# Let's check unique values
print("\nUnique values in CH2O:", sorted(df_clean['CH2O'].dropna().unique()))

# For MTRANS - check unique values
print("\nUnique values in MTRANS:", df_clean['MTRANS'].unique())

# For NObeyesdad (target) - check unique values
print("\nUnique values in NObeyesdad:", df_clean['NObeyesdad'].unique())

# 4. Let's look at the data more carefully
print("\nFirst few rows with issues:")
# Find rows with any missing values
rows_with_missing = df_clean[df_clean.isnull().any(axis=1)]
print(rows_with_missing.head())

# 5. Check for garbage values in numerical columns
numeric_cols = ['Age', 'Height', 'Weight', 'FCVC', 'NCP', 'CH2O', 'FAF', 'TUE']

print("\nChecking for extreme/unrealistic values:")
for col in numeric_cols:
    if col in df_clean.columns:
        print(f"\n{col}:")
        print(f"  Min: {df_clean[col].min()}")
        print(f"  Max: {df_clean[col].max()}")
        print(f"  Mean: {df_clean[col].mean():.2f}")
        print(f"  Missing: {df_clean[col].isnull().sum()}")

# 6. Check Age specifically - some values look like decimals (e.g., 25.196214)
# This might be intentional (e.g., age in years with decimal), but let's verify
age_stats = df_clean['Age'].describe()
print(f"\nAge statistics:\n{age_stats}")

# Check if there are ages that seem unrealistic (e.g., < 10 or > 100)
unrealistic_age = df_clean[(df_clean['Age'] < 10) | (df_clean['Age'] > 100)]
print(f"\nRows with Age < 10 or > 100: {len(unrealistic_age)}")

# 7. Check Weight and Height for unrealistic values
# Assuming adult dataset, height in meters
unrealistic_height = df_clean[(df_clean['Height'] < 1.0) | (df_clean['Height'] > 2.5)]
print(f"\nRows with Height < 1.0m or > 2.5m: {len(unrealistic_height)}")

unrealistic_weight = df_clean[(df_clean['Weight'] < 30) | (df_clean['Weight'] > 250)]
print(f"Rows with Weight < 30kg or > 250kg: {len(unrealistic_weight)}")
if len(unrealistic_weight) > 0:
    print("Extreme weight values:", df_clean.loc[unrealistic_weight.index, 'Weight'].unique()[:10])

# 8. Clean the data step by step
# Remove rows with missing target variable
df_clean = df_clean.dropna(subset=['NObeyesdad'])

# Fill missing categorical values with mode
categorical_cols = ['Gender', 'CALC', 'FAVC', 'SCC', 'SMOKE', 'CAEC', 'MTRANS', 'family_history_with_overweight']
for col in categorical_cols:
    if col in df_clean.columns:
        mode_val = df_clean[col].mode()[0] if not df_clean[col].mode().empty else 'Unknown'
        df_clean[col] = df_clean[col].fillna(mode_val)

# Fill missing numerical values with median (less sensitive to outliers)
for col in numeric_cols:
    if col in df_clean.columns and col != 'Age':  # We'll handle Age separately
        median_val = df_clean[col].median()
        df_clean[col] = df_clean[col].fillna(median_val)

# For Age, if missing, use median but round to nearest whole number
if 'Age' in df_clean.columns:
    median_age = df_clean['Age'].median()
    df_clean['Age'] = df_clean['Age'].fillna(median_age)

# 9. Convert Age to integer (since age is typically whole number)
# But first check if decimals are meaningful
print(f"\nAge values with decimals: {len(df_clean[df_clean['Age'] % 1 != 0])}")
# Given the nature of age data, let's round to nearest whole number
df_clean['Age'] = df_clean['Age'].round().astype(int)

# 10. Clean Gender column - standardize values
gender_mapping = {'M': 'Male', 'F': 'Female', 'male': 'Male', 'female': 'Female'}
df_clean['Gender'] = df_clean['Gender'].replace(gender_mapping)
df_clean['Gender'] = df_clean['Gender'].str.title()  # Ensure proper case

# 11. Clean other categorical columns
# Standardize yes/no columns
yes_no_cols = ['FAVC', 'SCC', 'SMOKE', 'family_history_with_overweight']
for col in yes_no_cols:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].str.lower().replace({'yes': 'yes', 'no': 'no', 'y': 'yes', 'n': 'no'})

# Clean CALC and CAEC columns
if 'CALC' in df_clean.columns:
    df_clean['CALC'] = df_clean['CALC'].str.title().replace({
        'Sometimes': 'Sometimes',
        'Frequently': 'Frequently',
        'Always': 'Always',
        'No': 'no'
    })

if 'CAEC' in df_clean.columns:
    df_clean['CAEC'] = df_clean['CAEC'].str.title().replace({
        'Sometimes': 'Sometimes',
        'Frequently': 'Frequently',
        'Always': 'Always',
        'No': 'no'
    })

# 12. Check for and remove extreme outliers in Height and Weight
# Calculate IQR for weight
Q1 = df_clean['Weight'].quantile(0.25)
Q3 = df_clean['Weight'].quantile(0.75)
IQR = Q3 - Q1
weight_lower_bound = Q1 - 1.5 * IQR
weight_upper_bound = Q3 + 1.5 * IQR

print(f"\nWeight IQR bounds: {weight_lower_bound:.1f} to {weight_upper_bound:.1f}")
outliers_weight = df_clean[(df_clean['Weight'] < weight_lower_bound) | (df_clean['Weight'] > weight_upper_bound)]
print(f"Potential weight outliers (IQR method): {len(outliers_weight)}")

# Calculate IQR for height
Q1_h = df_clean['Height'].quantile(0.25)
Q3_h = df_clean['Height'].quantile(0.75)
IQR_h = Q3_h - Q1_h
height_lower_bound = Q1_h - 1.5 * IQR_h
height_upper_bound = Q3_h + 1.5 * IQR_h

print(f"Height IQR bounds: {height_lower_bound:.2f} to {height_upper_bound:.2f}")
outliers_height = df_clean[(df_clean['Height'] < height_lower_bound) | (df_clean['Height'] > height_upper_bound)]
print(f"Potential height outliers (IQR method): {len(outliers_height)}")

# Instead of removing all outliers, let's cap extreme values for weight
# But check first - there's a weight of 173kg and 165kg which might be valid for obesity dataset
print(f"\nTop 5 highest weights: {sorted(df_clean['Weight'].unique(), reverse=True)[:5]}")
print(f"Top 5 lowest weights: {sorted(df_clean['Weight'].unique())[:5]}")

# Given this is an obesity dataset, extreme weights might be valid
# Let's only remove completely unrealistic values
# Keep weights between 30kg and 250kg (reasonable range for humans)
df_clean = df_clean[(df_clean['Weight'] >= 30) & (df_clean['Weight'] <= 250)]
df_clean = df_clean[(df_clean['Height'] >= 1.0) & (df_clean['Height'] <= 2.5)]

# 13. Check for inconsistent data
# Calculate BMI from Height and Weight to check against NObeyesdad
df_clean['BMI_calculated'] = df_clean['Weight'] / (df_clean['Height'] ** 2)

# Create a function to categorize BMI
def categorize_bmi(bmi):
    if bmi < 18.5:
        return 'Insufficient_Weight'
    elif 18.5 <= bmi < 25:
        return 'Normal_Weight'
    elif 25 <= bmi < 30:
        return 'Overweight_Level_I'
    elif 30 <= bmi < 35:
        return 'Overweight_Level_II'
    elif 35 <= bmi < 40:
        return 'Obesity_Type_I'
    elif 40 <= bmi < 45:
        return 'Obesity_Type_II'
    else:
        return 'Obesity_Type_III'

df_clean['BMI_category'] = df_clean['BMI_calculated'].apply(categorize_bmi)

# Check consistency between calculated BMI category and NObeyesdad
inconsistent = df_clean[df_clean['BMI_category'] != df_clean['NObeyesdad']]
print(f"\nRows with inconsistent BMI categorization: {len(inconsistent)}")

# For a small number of inconsistencies, we might trust the calculated BMI
# but for now, let's just note this

# 14. Clean up the dataset - remove helper columns
df_clean = df_clean.drop(['BMI_calculated', 'BMI_category'], axis=1)

# 15. Reset index
df_clean = df_clean.reset_index(drop=True)

# Final check
print("\n" + "="*50)
print("CLEANING COMPLETE")
print("="*50)
print(f"Original shape: {df.shape}")
print(f"Cleaned shape: {df_clean.shape}")
print(f"Rows removed: {df.shape[0] - df_clean.shape[0]}")

print("\nMissing values in cleaned dataset:")
print(df_clean.isnull().sum())

print("\nData types in cleaned dataset:")
print(df_clean.dtypes)

print("\nSample of cleaned data (first 5 rows):")
print(df_clean.head())

# Save cleaned dataset
df_clean.to_csv('Cleaned_Obesity_Dataset.csv', index=False)
print("\nCleaned dataset saved as 'Cleaned_Obesity_Dataset.csv'")

Total duplicates: 23

Missing values after removing duplicates:
Age                                0
Gender                            11
Height                             0
Weight                             0
CALC                               0
FAVC                               0
FCVC                               0
NCP                                0
SCC                                0
SMOKE                              0
CH2O                              21
family_history_with_overweight    21
FAF                                0
TUE                                0
CAEC                               0
MTRANS                             0
NObeyesdad                         0
dtype: int64

Rows with missing values:

Column: Gender
Missing count: 11

Column: CH2O
Missing count: 21

Column: family_history_with_overweight
Missing count: 21

Unique values in CH2O: [np.float64(1.0), np.float64(1.000463), np.float64(1.000536), np.float64(1.000544), np.float64(1.000695), np.float64(1.